# CASE DE ANÁLISE DE DADOS

**Profissional Júnior em Business Intelligence**

João Pedro Castanha Tenório Nunes

Data Analytics e Visualização de Dados - ÔxeTech Academy

# Contexto
Você foi contratado como Analista de BI Júnior em um marketplace de eletrônicos. A diretoria deseja compreender melhor o desempenho das vendas, a rentabilidade dos produtos e o comportamento dos clientes para apoiar decisões estratégicas.

Para realizar a análise, você recebeu três bases de dados:

• Pedidos: registros de todas as vendas realizadas.

• Produtos: informações sobre os produtos disponíveis na plataforma.

• Clientes: dados cadastrais e histórico básico dos clientes.

# Objetivos do Case
O objetivo é explorar, tratar e analisar os dados para responder às perguntas de negócio propostas. As análises poderão ser realizadas em SQL, Python ou combinando as duas ferramentas. O Power BI deverá ser utilizado para a criação de visualizações e indicadores interativos.

#1. Primeiros Passos

In [9]:
# Importando Bibliotecas.

import pandas as pd
import sqlite3

In [10]:
# Caminho dos Arquivos CSV (Atualizado para usar os arquivos anexados):

pedidos_csv = "pedidos (1).csv"
produtos_csv = "produtos (1).csv"
clientes_csv = "clientes (1).csv"

In [11]:
# Lendo os CSVs.

df_pedidos = pd.read_csv(pedidos_csv)
df_produtos = pd.read_csv(produtos_csv)
df_clientes = pd.read_csv(clientes_csv)

In [12]:
# Criando a conexão com um banco de dados SQLite em memória.

conn = sqlite3.connect(":memory:")

In [13]:
# Salvando os DataFrames como tabelas no banco.

df_pedidos.to_sql("pedidos", conn, index=False, if_exists="replace")
df_produtos.to_sql("produtos", conn, index=False, if_exists="replace")
df_clientes.to_sql("clientes", conn, index=False, if_exists="replace")

500

In [14]:
# Essa função permite escrever uma consulta SQL e receber o resultado como uma tabela do pandas.

def consulta(sql):
    return pd.read_sql_query(sql, conn)

#2. Tratamento dos Dados

# Tratamento da Tabela - **`pedidos`**

In [15]:
# Visualizando o número de linhas e colunas da tabela "pedidos" utilizando o shape.

linhas = df_pedidos.shape[0]
colunas = df_pedidos.shape[1]

print(f"Número de linhas: {linhas}\nNúmero de colunas: {colunas}")

Número de linhas: 1000
Número de colunas: 9


In [16]:
# Visualizando as primeiras linhas da tabela "pedidos" utilizando o head.

df_pedidos.head()

,pedido_id,cliente_id,data_pedido,produto_id,quantidade,preco_unitario,status_pedido,custo_produto,metodo_pagamento
0,1001,684,2024-05-16,2035,1,3843.28,Concluído,431.06,PIX
1,1002,737,2024-10-22,2042,4,2891.61,Cancelado,1454.27,Cartão de Crédito
2,1003,770,2025-02-20,2029,1,3354.05,Concluído,270.50,PIX
3,1004,773,2024-04-08,2010,1,208.95,Cancelado,3451.46,Cartão de Crédito
4,1005,872,2024-12-19,2001,4,185.65,Concluído,581.70,Boleto


In [17]:
# Visualizando o Dtype das colunas registradas na tabela "pedidos" utilizando o info.

df_pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pedido_id         1000 non-null   int64  
 1   cliente_id        1000 non-null   int64  
 2   data_pedido       1000 non-null   object 
 3   produto_id        1000 non-null   int64  
 4   quantidade        1000 non-null   int64  
 5   preco_unitario    1000 non-null   float64
 6   status_pedido     1000 non-null   object 
 7   custo_produto     1000 non-null   float64
 8   metodo_pagamento  1000 non-null   object 
dtypes: float64(2), int64(4), object(3)
memory usage: 70.4+ KB


In [18]:
# Convertendo a coluna "data_pedido" para datetime.

df_pedidos["data_pedido"] = pd.to_datetime(df_pedidos["data_pedido"], errors="coerce")

In [19]:
# Convertendo as colunas "status_pedido" e "metodo_pagamento" para category.

df_pedidos["status_pedido"] = df_pedidos["status_pedido"].astype("category")
df_pedidos["metodo_pagamento"] = df_pedidos["metodo_pagamento"].astype("category")

In [20]:
# Verificando se as colunas "data_pedido", "status_pedido" e "metodo_pagamento" foram convertidas.

df_pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   pedido_id         1000 non-null   int64         
 1   cliente_id        1000 non-null   int64         
 2   data_pedido       1000 non-null   datetime64[ns]
 3   produto_id        1000 non-null   int64         
 4   quantidade        1000 non-null   int64         
 5   preco_unitario    1000 non-null   float64       
 6   status_pedido     1000 non-null   category      
 7   custo_produto     1000 non-null   float64       
 8   metodo_pagamento  1000 non-null   category      
dtypes: category(2), datetime64[ns](1), float64(2), int64(4)
memory usage: 57.1 KB


In [21]:
# Visualizando as estatísticas descritivas das colunas numéricas da tabela "pedidos" utilizando o describe.
# O round foi utilizado para arredondar os cálculos para duas casas decimais.

df_pedidos.describe().round(2)

,pedido_id,cliente_id,data_pedido,produto_id,quantidade,preco_unitario,custo_produto
count,1000.00,1000.00,1000,1000.00,1000.00,1000.00,1000.00
mean,1500.50,743.45,2024-08-30 23:34:04.799999744,2026.02,2.96,2578.65,2082.20
min,1001.00,501.00,2024-02-26 00:00:00,2001.00,1.00,106.66,52.13
25%,1250.75,619.00,2024-06-07 00:00:00,2014.00,2.00,1380.19,1053.31
50%,1500.50,738.50,2024-08-25 12:00:00,2026.00,3.00,2627.23,2181.18
75%,1750.25,868.00,2024-11-30 00:00:00,2039.00,4.00,3739.16,3058.55
max,2000.00,999.00,2025-02-25 00:00:00,2050.00,5.00,4993.64,3998.25
std,288.82,142.78,NaN,14.37,1.41,1369.73,1146.13


In [22]:
# Verificando se existem valores nulos na tabela "pedidos" utilizando o isna.
# O sum foi utilizado para somar o número de valores de cada coluna, caso existam.

df_pedidos.isna().sum()

,0
pedido_id,0
cliente_id,0
data_pedido,0
produto_id,0
quantidade,0
preco_unitario,0
status_pedido,0
custo_produto,0
metodo_pagamento,0


In [23]:
# Verificando se existem duplicatas na tabela "pedidos" utilizando o duplicated.
# O sum foi utilizado para somar o número de valores duplicados na tabela, caso existam.

df_pedidos.duplicated().sum()

np.int64(0)

In [24]:
df_pedidos

,pedido_id,cliente_id,data_pedido,produto_id,quantidade,preco_unitario,status_pedido,custo_produto,metodo_pagamento
0,1001,684,2024-05-16,2035,1,3843.28,Concluído,431.06,PIX
1,1002,737,2024-10-22,2042,4,2891.61,Cancelado,1454.27,Cartão de Crédito
2,1003,770,2025-02-20,2029,1,3354.05,Concluído,270.50,PIX
3,1004,773,2024-04-08,2010,1,208.95,Cancelado,3451.46,Cartão de Crédito
4,1005,872,2024-12-19,2001,4,185.65,Concluído,581.70,Boleto
...,...,...,...,...,...,...,...,...,...
995,1996,566,2025-02-24,2002,2,860.43,Concluído,2706.45,Boleto
996,1997,762,2024-10-13,2004,4,317.94,Cancelado,594.51,Transferência Bancária
997,1998,611,2024-11-28,2039,1,1870.06,Concluído,2875.82,Transferência Bancária
998,1999,635,2024-10-22,2029,4,3527.86,Concluído,96.15,Transferência Bancária


In [25]:
# Carregando a versão limpa da tabela "pedidos".

df_pedidos.to_excel("pedidos_limpos.xlsx", index=False)

## Feedback Geral sobre o Tratamento da Tabela `pedidos`

O tratamento da tabela `pedidos` foi eficaz e seguiu as boas práticas:

*   **Análise Inicial (shape, head, info):** Essencial para entender a estrutura, volume e tipos de dados.
*   **Tratamento de Tipos:** A coluna `data_pedido` foi convertida para `datetime` (importante para análises temporais), e `status_pedido` e `metodo_pagamento` para `category` (para otimização e consistência). `errors="coerce"` em datas é bom para evitar interrupções.
*   **Verificação de Qualidade de Dados:**
    *   **Valores Nulos:** Verificado e **NÃO foram encontrados valores nulos** em nenhuma coluna, o que é excelente para a qualidade dos dados.
    *   **Duplicatas:** Verificado e **NÃO foram encontradas duplicatas**, garantindo a unicidade dos registros.
*   **Estatísticas Descritivas:** Utilizadas para um rápido entendimento da distribuição das colunas numéricas.
*   **Salvamento:** A versão limpa foi salva em `pedidos_limpo.xlsx`, um passo crucial para manter o estado tratado e pronto para as próximas análises.

Em resumo, a tabela `pedidos` está bem preparada e consistente para ser integrada às análises de Business Intelligence.


Texto otimizado pelo Gemini*

# Tratamento da Tabela - **`produtos`**

In [26]:
# Visualizando o número de linhas e colunas da tabela "produtos" utilizando o shape.

linhas = df_produtos.shape[0]
colunas = df_produtos.shape[1]

print(f"Número de linhas: {linhas}\nNúmero de colunas: {colunas}")

Número de linhas: 50
Número de colunas: 5


In [27]:
# Visualizando as primeiras linhas da tabela "produtos" utilizando o head.

df_produtos.head()

,produto_id,nome_produto,categoria,fornecedor,estoque_disponivel
0,2001,Produto 2001,Acessórios,Wood-Barber,64
1,2002,Produto 2002,Tablets,Fields and Sons,82
2,2003,Produto 2003,Notebooks,Henry-Brown,55
3,2004,Produto 2004,Acessórios,"Hernandez, Pham and Moreno",95
4,2005,Produto 2005,Câmeras,Williams-Garcia,84


In [28]:
# Visualizando o Dtype das colunas registradas na tabela "produtos" utilizando o info.

df_produtos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   produto_id          50 non-null     int64 
 1   nome_produto        50 non-null     object
 2   categoria           50 non-null     object
 3   fornecedor          50 non-null     object
 4   estoque_disponivel  50 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.1+ KB


In [29]:
# Convertendo a coluna "categoria" para category.

df_produtos["categoria"] = df_produtos["categoria"].astype("category")

In [30]:
# Verificando se a coluna "categoria" foi convertida.

df_produtos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   produto_id          50 non-null     int64   
 1   nome_produto        50 non-null     object  
 2   categoria           50 non-null     category
 3   fornecedor          50 non-null     object  
 4   estoque_disponivel  50 non-null     int64   
dtypes: category(1), int64(2), object(2)
memory usage: 2.1+ KB


In [31]:
# Visualizando as estatísticas descritivas das colunas numéricas da tabela "produtos" utilizando o describe.
# O round foi utilizado para arredondar os cálculos para duas casas decimais.

df_produtos.describe().round(2)

,produto_id,estoque_disponivel
count,50.00,50.00
mean,2025.50,60.20
std,14.58,28.60
min,2001.00,11.00
25%,2013.25,36.50
50%,2025.50,63.50
75%,2037.75,88.75
max,2050.00,99.00


In [32]:
# Verificando se existem valores nulos na tabela "produtos" utilizando o isna.
# O sum foi utilizado para somar o número de valores nulos de cada coluna, caso existam.

df_produtos.isna().sum()

,0
produto_id,0
nome_produto,0
categoria,0
fornecedor,0
estoque_disponivel,0


In [33]:
# Verificando se existem duplicatas na tabela "produtos" utilizando o duplicated.
# O sum foi utilizado para somar o número de valores duplicados, caso existam.

df_produtos.duplicated().sum()

np.int64(0)

In [34]:
df_produtos

,produto_id,nome_produto,categoria,fornecedor,estoque_disponivel
0,2001,Produto 2001,Acessórios,Wood-Barber,64
1,2002,Produto 2002,Tablets,Fields and Sons,82
2,2003,Produto 2003,Notebooks,Henry-Brown,55
3,2004,Produto 2004,Acessórios,"Hernandez, Pham and Moreno",95
4,2005,Produto 2005,Câmeras,Williams-Garcia,84
5,2006,Produto 2006,Tablets,Carpenter Ltd,40
6,2007,Produto 2007,Smartwatches,"Williams, Preston and Smith",41
7,2008,Produto 2008,Celulares,"Fox, Gutierrez and Johnson",92
8,2009,Produto 2009,Monitores,Hernandez-Cole,94
9,2010,Produto 2010,Consoles,Carlson-Hunter,15


In [35]:
# Carregando a versão limpa da tabela "produtos".

df_produtos.to_csv("produtos_limpo.csv", index=False)

## Feedback Geral sobre o Tratamento da Tabela `produtos`

O tratamento da tabela `produtos` foi direto e eficaz, seguindo as melhores práticas para garantir a qualidade dos dados:

*   **Análise Inicial (shape, head, info):** Fundamental para compreender a dimensão, amostra e tipos de dados da tabela.
*   **Tratamento de Tipos:** A coluna `categoria` foi convertida para `category`, o que é benéfico para otimização de memória e para análises futuras de dados categóricos. Como os valores das colunas `nome_produto` e `fornecedor` são variados (não seguem um padrão, como as categorias), não foi necessário realizar a conversão para `category`, visto que não teria vantagem alguma.
*   **Verificação de Qualidade de Dados:**
    *   **Valores Nulos:** Verificado e **NÃO foram encontrados valores nulos** em nenhuma coluna, indicando um conjunto de dados limpo nesse aspecto.
    *   **Duplicatas:** Verificado e **NÃO foram encontradas duplicatas**, confirmando a unicidade dos registros de produtos.
*   **Estatísticas Descritivas:** Utilizadas para um rápido panorama das colunas numéricas, como `estoque_disponivel`.
*   **Salvamento:** A tabela tratada foi salva como `produtos_limpo.csv`, assegurando que a versão limpa esteja disponível para as próximas etapas.

Em resumo, a tabela `produtos` está bem preparada e consistente para ser integrada às análises de Business Intelligence.

Texto otimizado pelo Gemini*

# Tratamento da Tabela - `clientes`

In [36]:
# Visualizando o número de linhas e colunas da tabela "clientes" utilizando o shape.

linhas = df_clientes.shape[0]
colunas = df_clientes.shape[1]

print(f"Número de linhas: {linhas}\nNúmero de colunas: {colunas}")

Número de linhas: 500
Número de colunas: 6


In [37]:
# Visualizando as primeiras linhas da tabela "clientes" utilizando o head.

df_clientes.head()

,cliente_id,nome_cliente,cidade,estado,data_cadastro,total_pedidos
0,501,Patrick Fernandez,Lake Jack,WA,2025-02-21,5
1,502,Kristin Weiss,Wadehaven,FL,2023-03-21,4
2,503,Kenneth Sanchez,North Michael,NH,2024-12-31,4
3,504,Lisa Martin,New Danny,CO,2023-11-14,2
4,505,Jennifer Hall,Harrisonburgh,GA,2024-05-07,3


In [38]:
# Visualizando o Dtype das colunas registradas na tabela "clientes" utilizando o info.

df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cliente_id     500 non-null    int64 
 1   nome_cliente   500 non-null    object
 2   cidade         500 non-null    object
 3   estado         500 non-null    object
 4   data_cadastro  500 non-null    object
 5   total_pedidos  500 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 23.6+ KB


In [39]:
# Convertendo a coluna "data_cadastro" para datetime.

df_clientes["data_cadastro"] = pd.to_datetime(df_clientes["data_cadastro"], errors="coerce")

In [40]:
# Verificando se a coluna "data_cadastro" foi convertida.

df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   cliente_id     500 non-null    int64         
 1   nome_cliente   500 non-null    object        
 2   cidade         500 non-null    object        
 3   estado         500 non-null    object        
 4   data_cadastro  500 non-null    datetime64[ns]
 5   total_pedidos  500 non-null    int64         
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 23.6+ KB


In [41]:
# Visualizando as estatísticas descritivas das colunas numéricas da tabela "clientes" utilizando o describe.
# O round foi utilizado para arredondar os cálculos para duas casas decimais.

df_clientes.describe().round(2)

,cliente_id,data_cadastro,total_pedidos
count,500.00,500,500.00
mean,750.50,2024-03-01 00:28:48,2.94
min,501.00,2023-03-01 00:00:00,1.00
25%,625.75,2023-08-27 18:00:00,2.00
50%,750.50,2024-02-29 12:00:00,3.00
75%,875.25,2024-08-24 00:00:00,4.00
max,1000.00,2025-02-25 00:00:00,5.00
std,144.48,NaN,1.44


In [42]:
# Verificando se existem valores nulos na tabela "clientes" utilizando o isna.
# O sum foi utilizado para somar o número de valores nulos de cada coluna, caso existam.

df_clientes.isna().sum()

,0
cliente_id,0
nome_cliente,0
cidade,0
estado,0
data_cadastro,0
total_pedidos,0


In [43]:
# Verificando se existem duplicatas na tabela "clientes" utilizando o duplicated.
# O sum foi utilizado para somar o número de valores duplicados, caso existam.

df_clientes.duplicated().sum()

np.int64(0)

In [44]:
df_clientes

,cliente_id,nome_cliente,cidade,estado,data_cadastro,total_pedidos
0,501,Patrick Fernandez,Lake Jack,WA,2025-02-21,5
1,502,Kristin Weiss,Wadehaven,FL,2023-03-21,4
2,503,Kenneth Sanchez,North Michael,NH,2024-12-31,4
3,504,Lisa Martin,New Danny,CO,2023-11-14,2
4,505,Jennifer Hall,Harrisonburgh,GA,2024-05-07,3
...,...,...,...,...,...,...
495,996,Brian Lewis,Port Christy,PA,2024-03-06,3
496,997,Christina Watson,West Jenniferview,AZ,2023-09-29,5
497,998,Melissa Watson,Colinshire,MO,2024-02-20,3
498,999,Jessica Roman,New Chelsea,MN,2024-11-17,3


In [45]:
# Carregando a versão limpa da tabela "clientes".

df_clientes.to_csv("clientes_limpo.csv", index=False)

## Feedback Geral sobre o Tratamento da Tabela `clientes`

O tratamento da tabela `clientes` foi realizado de forma eficiente, garantindo a qualidade e a prontidão dos dados para análises futuras:

*   **Análise Inicial (shape, head, info):** Essencial para compreender a estrutura, volume e os tipos de dados contidos na tabela.
*   **Tratamento de Tipos:** A coluna `data_cadastro` foi convertida para o tipo `datetime`, o que é fundamental para qualquer análise temporal de cadastro de clientes. As colunas "nome_cliente", "estado" e "cidade" possuem valores variados (não seguem um padrão), por isso, não foram convertidas para `category`, visto que não teria vantagem alguma.
*   **Verificação de Qualidade de Dados:**
    *   **Valores Nulos:** Foi verificado que **NÃO existem valores nulos** em nenhuma coluna, o que é um excelente indicativo da integridade dos dados.
    *   **Duplicatas:** Foi confirmado que **NÃO há registros duplicados**, assegurando a unicidade de cada cliente.
*   **Estatísticas Descritivas:** Utilizadas para obter um panorama rápido das colunas numéricas, como `cliente_id` e `total_pedidos`.
*   **Salvamento:** A versão limpa da tabela foi salva como `clientes_limpo.csv`, o que é um passo importante para preservar os dados tratados e permitir fácil acesso nas próximas etapas.

Em suma, a tabela `clientes` está devidamente preparada e consistente, pronta para ser utilizada em integrações e análises de Business Intelligence.

Texto otimizado pelo Gemini*

# 3. Desafios e Perguntas a serem respondidas:

In [46]:
# Lendo as Tabelas.

pedidos_limpos = pd.read_excel("pedidos_limpos.xlsx")
produtos_limpo = pd.read_csv("produtos_limpo.csv")
clientes_limpo = pd.read_csv("clientes_limpo.csv")

In [47]:
# Salvando os DataFrames como tabelas no banco.

pedidos_limpos.to_sql("pedidos_limpos", conn, index=False, if_exists="replace")
produtos_limpo.to_sql("produtos_limpo", conn, index=False, if_exists="replace")
clientes_limpo.to_sql("clientes_limpo", conn, index=False, if_exists="replace")

500

# 1. Qual foi o faturamento mensal do marketplace no período analisado?

In [48]:
# Faturamento mensal do marketplace no período analisado:

consulta("""
SELECT
    strftime('%Y-%m', data_pedido) AS mes,
    SUM(quantidade * preco_unitario) AS faturamento_mensal
FROM pedidos_limpos
WHERE status_pedido = 'Concluído'
GROUP BY mes
ORDER BY faturamento_mensal DESC;
""")

,mes,faturamento_mensal
0,2024-07,422559.85
1,2024-06,411797.91
2,2024-12,396642.38
3,2025-01,376026.03
4,2024-08,320931.12
5,2024-04,315860.70
6,2024-10,304837.81
7,2024-03,302697.09
8,2024-09,281829.77
9,2025-02,236952.65


* **Principais Achados:**

  * **Picos de Receita em 2024**: Julho (422.559,85) e Junho (411.797,91) foram os meses mais fortes, indicando um bom faturamento no meio do ano.

  * **Queda Gradual em 2025**: Fevereiro (236.952,65) em relação à Janeiro (376.026,03) mostra uma redução de aproximadamente 37% do faturamento, valor significativo.

  * **Meses de Menor Desempenho**: Fevereiro/2024 (6.134,37) e Maio/2024 (185.763,13) tiveram faturamento muito abaixo da média, foram períodos críticos.

  * **Tendência Geral**: Forte crescimento no segundo semestre de 2024, seguido por retração no início de 2025.

* **Recomendações Estratégicas**:

* **Explorar os meses de Junho e Julho**
  * Realizar investimentos em campanhas de marketing e promoções, aproveitando o histórico de alta demanda nesses meses.
  * Planejar estoque e logística para suportar o pico de vendas.

* **Queda entre Janeiro e Fevereiro**
  * Avaliar a motivação da queda, se foram motivos internos ou externos.
  * Garantir um bom estoque para que os produtos que mais são vendidos em Janeiro estejam disponíveis nos demais meses.
  * Criar ações de marketing voltadas ao início do ano (bons descontos).

* **Analisar os meses críticos**
  * Fevereiro/2024 foi um ponto fora da curva. Investigar causas: problemas operacionais, baixa oferta de produtos ou falhas em campanhas.
  * Ajustar estratégias para evitar repetição de quedas tão acentuadas.

* **Diversificar produtos e canais**
  * Ampliar mix de produtos nos meses de baixa demanda.
  * Explorar novos canais de venda (marketplaces parceiros, redes sociais, campanhas segmentadas).

* **Monitoramento Contínuo**
  * Criar dashboards mensais de faturamento para acompanhar tendências em tempo real.
  * Usar análises preditivas para antecipar quedas e agir preventivamente.


# 2. Qual foi o lucro mensal do marketplace no período analisado?

In [49]:
# Lucro mensal do marketplace no período analisado:

consulta("""
SELECT
    strftime('%Y-%m', data_pedido) AS mes,
    SUM (quantidade * (preco_unitario - custo_produto)) as lucro_mensal
FROM pedidos_limpos
WHERE status_pedido = "Concluído"
GROUP BY mes
ORDER BY lucro_mensal DESC;
""")

,mes,lucro_mensal
0,2024-07,161477.99
1,2024-06,109026.12
2,2024-03,90086.86
3,2025-02,86790.47
4,2024-12,78818.32
5,2024-08,72893.98
6,2024-04,59489.96
7,2025-01,48684.40
8,2024-11,27189.01
9,2024-09,22639.32


* **Principais Achados**
  * **Maiores Lucros em 2024**: Julho (161.477,99) e Junho (109.026,12) foram os meses mais fortes, indicando um bom lucro no meio do ano.

  * **Prejuízos em 2024**: Outubro (-28.532,93), Maio (-4.441,10) e Fevereiro (-3.864,15) foram os meses em que os custos superaram as receitas.

  * **Queda Acentuada no início de 2025**: Houve uma queda considerável entre o fim de 2024 e começo de 2025, em que Janeiro/2025 (48.684,40) teve uma queda de aproximadamente 32% em relação à Dezembro/2024 (78.818,32). Porém, em Fevereiro/2025 (86.790,47) houve um aumento significativo de aproximadamente 44% em relação à Janeiro.

  * **Volatilidade**: Há grande variação entre meses de lucro elevado e meses de prejuízo, indicando falta de estabilidade financeira.

* **Recomendações Estratégicas**
* **Explorar os meses de Junho e Julho**
  * Planejar campanhas de reforço de estoque nesses meses, visto que historicamente apresentam maior lucratividade.
  * Replicar estratégias utilizadas anteriormente nesse período.

* **Analisar os meses críticos**
  * Outubro, Maio e Fevereiro/2024 foram pontos fora da curva, há a necessidade de investigar as causas dos prejuízos.
  * De mandeira geral, mudar a estratégia que foi utilizada nesses meses (campanhas, preços, produtos).

* **Fortalecimento no início do ano**
  * Estratégias sazonais são essenciais, campanhas que escalam em eventos de início de ano (pós-férias, início das aulas, volta aos trabalhos, novo ciclo).

* **Reduzir Volatilidade**
  * Criar estratégias para manter estabilidade durante todo o ano: melhorar campanhas, diversificar produtos, fortalecer os canais de venda.

* **Monitoramento Contínuo**
  * Criar dashboards mensais de lucro para acompanhar tendências em tempo real.
  * Usar análises preditivas para antecipar quedas e agir preventivamente.


# 3. Qual foi a margem de lucro média mensal, considerando o custo dos produtos?

In [50]:
# Margem de lucro média mensal, considerando o custo dos produtos:

consulta("""
SELECT
    strftime('%Y-%m', data_pedido) AS mes,
    ROUND(
        SUM((quantidade * preco_unitario) - (quantidade * custo_produto))
        / SUM(quantidade * preco_unitario) * 100, 2
    ) AS margem_lucro_percentual
FROM pedidos_limpos
WHERE status_pedido = 'Concluído'
GROUP BY mes
ORDER BY margem_lucro_percentual DESC;
""")

,mes,margem_lucro_percentual
0,2024-07,38.21
1,2025-02,36.63
2,2024-03,29.76
3,2024-06,26.48
4,2024-08,22.71
5,2024-12,19.87
6,2024-04,18.83
7,2025-01,12.95
8,2024-11,12.25
9,2024-09,8.03


* **Principais Achados**
  * **Maiores margens de lucro**: Julho/2024 (38,21%) e Fevereiro/2025 (36,63%) foram os meses mais eficientes, com alta capacidade de transformar faturamento em lucro.

  * **Margens negativas**: Fevereiro/2024 (-62,99%), Outubro/2024 (-9,36%) e Maio/2024 (-2,39%) revelam meses críticos em que os custos superaram as receitas, com destaque para Fevereiro, em que a margem percentual foi muito alta.

  * **Margens baixas**: Alguns meses como Setembro/2024 (8,03%), Novembro/2024 (12,25%) e Janeiro/2025 (12,95%) possuíram margens de lucro baixas.

  * **Volatilidade acentuada**: A margem varia de -62,99% até 38,21%, mostrando falta de estabilidade operacional e financeira.

* **Recomendações Estrategicas**

* **Replicar práticas dos melhores meses**
  * Analisar o que funcionou em Julho/2024 e Fevereiro/2025 (disponibilidade de produtos, campanhas, preços) e aplicar em outros períodos.

* **Analisar meses de margem negativa**
  * Investigar as causas dos péssimos resultados obtidos em Fevereiro/2024 (principalmente), Outubro/2024 e Maio/2024.
  * Ajustar preços e diminuir custos nesses períodos.

* **Reduzir Volatilidade**
  * Criar estratégias para manter margens mais estáveis ao longo do ano.
  * Diversificar produtos e fortalecer os canais de vendas durante todo o ano de acordo com festividades, eventos, feriados.

* **Fortalecer meses com margens baixas**
  * Melhorar eficiência operacional e renegociar custos para aumentar a rentabilidade.

# 4. Qual é a taxa de cancelamento mensal de pedidos?

In [51]:
# Taxa de cancelamento mensal de pedidos:

consulta("""
SELECT
    strftime('%Y-%m', data_pedido) AS mes,
    ROUND(
        (COUNT(CASE WHEN status_pedido = 'Cancelado' THEN 1 END) * 100.0)
        / COUNT(*), 2
    ) AS taxa_cancelamento_percentual
FROM pedidos_limpos
GROUP BY mes
ORDER BY taxa_cancelamento_percentual DESC;
""")

,mes,taxa_cancelamento_percentual
0,2024-02,80.00
1,2024-08,56.67
2,2024-05,55.26
3,2024-11,52.70
4,2024-06,51.65
5,2024-09,50.79
6,2024-04,50.59
7,2024-07,49.48
8,2024-10,48.00
9,2025-02,47.89


*   **Principais Achados**
    * **Pico em Fevereiro/2024**: O mês de Fevereiro/2024 apresenta uma taxa de cancelamento excepcionalmente alta (80%), sendo um ponto fora da curva que necessita de investigação aprofundada.

    * **Altas taxas**: A maioria dos meses registra taxas de cancelamento acima de 40%, com vários meses superando 50%, o que indica um problema sistêmico na jornada do cliente ou na gestão de pedidos.
    
    * **Meses de melhor desempenho**: Janeiro/2025 (41,57%) e Março/2024 (45,71%) foram os meses com as menores taxas, mostrando que é possível ter um desempenho melhor.
    
    * **Volatilidade**: Há uma grande volatilidade nas taxas de cancelamento ao longo do período, variando de 41,57% a 80%, o que sugere inconstância nos processos ou fatores externos.

*   **Recomendações Estratégicas**
    
* **Investigar Fevereiro/2024**
    * Realizar uma análise detalhada dos pedidos cancelados em Fevereiro/2024 para identificar a causa raiz (problemas de estoque, falha no sistema, campanhas enganosas, atrasos na entrega).
    
* **Redução proativa**
    * Implementar estratégias para reduzir as altas taxas de cancelamento, como: descrições de produtos mais claras, informações de estoque em tempo real, melhoria na logística de entrega e aprimoramento do atendimento ao cliente.
    
* **Análise das causas de cancelamento**
    * Coletar feedback dos clientes e analisar os motivos mais comuns de cancelamento para identificar padrões e pontos de melhoria (ex: problemas com produtos específicos, regiões, métodos de pagamento).
    
* **Monitoramento contínuo**
    * Criar e acompanhar dashboards em tempo real para a taxa de cancelamento, permitindo intervenções rápidas em caso de anomalias.
    
* **Experiência Pós-Compra**
    * Fortalecer a comunicação pós-compra e oferecer incentivos para garantir a conclusão dos pedidos, melhorando a satisfação do cliente e a retenção.

# 5. Quais são os três produtos mais vendidos e seus respectivos faturamentos?

In [52]:
# Três produtos mais vendidos e seus respectivos faturamentos:

consulta("""
SELECT
    nome_produto,
    SUM(quantidade) AS quantidade_vendida,
    SUM(quantidade * preco_unitario) AS faturamento
FROM pedidos_limpos
JOIN produtos_limpo ON pedidos_limpos.produto_id = produtos_limpo.produto_id
WHERE status_pedido = 'Concluído'
GROUP BY nome_produto
ORDER BY quantidade_vendida DESC
LIMIT 3;
""")

,nome_produto,quantidade_vendida,faturamento
0,Produto 2034,61,167730.42
1,Produto 2019,56,163951.82
2,Produto 2014,52,157427.16


In [53]:
# Dez produtos mais vendidos e seus respectivos faturamentos:

consulta("""
SELECT
    nome_produto,
    SUM(quantidade) AS quantidade_vendida,
    SUM(quantidade * preco_unitario) AS faturamento
FROM pedidos_limpos
JOIN produtos_limpo ON pedidos_limpos.produto_id = produtos_limpo.produto_id
WHERE status_pedido = 'Concluído'
GROUP BY nome_produto
ORDER BY quantidade_vendida DESC
LIMIT 10;
""")

,nome_produto,quantidade_vendida,faturamento
0,Produto 2034,61,167730.42
1,Produto 2019,56,163951.82
2,Produto 2014,52,157427.16
3,Produto 2023,48,114657.30
4,Produto 2020,48,112678.19
5,Produto 2036,47,81146.32
6,Produto 2004,46,81966.38
7,Produto 2032,44,92626.03
8,Produto 2026,44,86441.10
9,Produto 2013,41,104200.06


*   **Principais Achados**
    * **Consistência dos produtos**: Os três produtos mais vendidos (Produto 2034, Produto 2019 e Produto 2014) demonstram um desempenho robusto tanto em quantidade vendida quanto em faturamento, indicando uma alta demanda por esses itens específicos.

    * **Faturamento elevado**: O "Produto 2034" se destaca com o maior faturamento (167.730,42) e maior quantidade vendida (61 unidades), consolidando-o como o principal gerador de receita entre os top 3.

    * **Próximos concorrentes**: "Produto 2019" e "Produto 2014" também apresentam faturamento e quantidade vendida expressivos e muito próximos, sugerindo que há um grupo de produtos de alto valor que impulsiona as vendas.

    * **Curiosidade**: Como é possível ver na consulta acima, nem sempre o produto mais vendido é o que gera mais faturamento.

*   **Recomendações Estratégicas**
* **Foco nos 3 mais vendidos**
  * Priorizar o estoque e a disponibilidade desses três produtos, garantindo que nunca faltem para os clientes, pois são os que mais contribuem para o faturamento.

* **Campanhas de marketing direcionadas**
  * Criar campanhas de marketing específicas para esses produtos, destacando seus diferenciais e popularidade para atrair ainda mais vendas.

* **Análise de margem de lucro**
  * Avaliar a margem de lucro individual de cada um desses produtos. Se a margem for alta, pode-se considerar investir ainda mais em sua promoção, se for mais baixa, buscar otimizar custos ou estratégias de precificação.

* **Expansão de variedade**
  * Estudar as características desses produtos de sucesso para identificar tendências e desenvolver ou adquirir produtos similares que possam replicar seu desempenho.

* **Programas de fidelidade**
  * Oferecer promoções ou programas de fidelidade para clientes que compram esses produtos frequentemente, incentivando a recompra e fortalecendo o relacionamento.

# 6. Quais são os métodos de pagamento mais utilizados?

In [54]:
# Métodos de pagamento mais utilizados:

consulta("""
SELECT
    metodo_pagamento,
    COUNT(*) AS quantidade_metodo
FROM pedidos_limpos
GROUP BY metodo_pagamento
ORDER BY quantidade_metodo DESC;
""")

,metodo_pagamento,quantidade_metodo
0,Transferência Bancária,267
1,PIX,256
2,Cartão de Crédito,248
3,Boleto,229


In [55]:
# Métodos de pagamento mais utilizados apenas em pedidos concluídos:

consulta("""
SELECT
    metodo_pagamento,
    COUNT(*) AS quantidade_metodo
FROM pedidos_limpos
WHERE status_pedido = "Concluído"
GROUP BY metodo_pagamento
ORDER BY quantidade_metodo DESC;
""")

,metodo_pagamento,quantidade_metodo
0,Transferência Bancária,137
1,PIX,124
2,Cartão de Crédito,123
3,Boleto,117


*   **Principais Achados**
    *   **Consistência nos métodos**: A "Transferência Bancária" e o "PIX" são consistentemente os métodos de pagamento mais utilizados, tanto no total de pedidos quanto nos pedidos concluídos, indicando forte preferência dos clientes por essas opções.

    *   **Proximidade de preferência**: O "Cartão de Crédito" e o "Boleto" também são bastante usados, com volumes próximos entre si, mas ligeiramente abaixo dos dois primeiros.

    *   **Padrão de conclusão**: A ordem de preferência dos métodos de pagamento permanece a mesma tanto para pedidos totais quanto para pedidos concluídos, sugerindo que o método de pagamento em si não é o principal fator de cancelamento após a escolha.

    *   **Potencial de melhorias**: Todos os métodos de pagamento têm uma taxa de conclusão de aproximadamente 50%, indicando um espaço significativo para melhorar as taxas de conversão e reduzir os cancelamentos em geral, independentemente do método escolhido.


*   **Recomendações Estratégicas**
* **Melhoria dos métodos preferidos**
  * Investir na otimização contínua da experiência de "Transferência Bancária" e "PIX", garantindo que o processo seja o mais rápido e sem atritos possível para capitalizar a preferência do cliente.

* **Análise de fluxo de cancelamento**
  * Aprofundar a análise na taxa de cancelamento de 50% para todos os métodos. Entender os motivos do cancelamento após a seleção do pagamento (ex: problemas no processo de checkout, falta de estoque, tempo de aprovação) para reduzir a perda de vendas.

* **Incentivos para conclusão**
  * Considerar oferecer pequenos incentivos ou lembretes para a conclusão de pagamentos via "Boleto" e "Cartão de Crédito", especialmente para reduzir o abandono de carrinho.

* **Segurança e confiança**
  * Reforçar a segurança e a transparência em todos os métodos de pagamento para aumentar a confiança do cliente e diminuir a hesitação na finalização da compra.


# 7. Quais estados geraram mais vendas?

In [56]:
# Estados que geraram mais vendas:

consulta("""
SELECT
    estado,
    COUNT(*) AS quantidade_vendas
FROM pedidos_limpos
JOIN clientes_limpo ON pedidos_limpos.cliente_id = clientes_limpo.cliente_id
WHERE status_pedido = 'Concluído'
GROUP BY estado
ORDER BY quantidade_vendas DESC;
""")

,estado,quantidade_vendas
0,MT,21
1,SC,19
2,HI,16
3,CT,16
4,MD,15
5,WI,14
6,NC,14
7,MS,14
8,ME,14
9,PA,13


*   **Principais Achados**
    *   **Concentração geográfica**: As vendas estão concentradas em poucos estados, com MT (21), SC (19), HI (16) e CT (16) liderando o número de vendas, indicando onde o marketplace tem maior penetração.

    *   **Disparidade de desempenho**: Há uma grande disparidade entre os estados, onde alguns contribuem significativamente para o volume total de vendas, enquanto outros, como ND (2 vendas), KS (4 vendas) e WY (4 vendas), têm uma participação mínima.

    *   **Oportunidade de expansão**: A vasta quantidade de estados com poucas vendas aponta para um potencial inexplorado e a necessidade de estratégias de marketing e vendas mais direcionadas para expandir a base de clientes nessas regiões.

    *   **Presença Nacional com variação**: Embora o marketplace tenha alcance nacional (vendas em diversos estados), a eficiência da distribuição e o reconhecimento da marca variam drasticamente por região.

*   **Recomendações Estratégicas**
* **Reforçar Etados com melhor performance**
  * Investir em marketing e logística nos estados de maior volume (MT, SC, HI, CT) para manter a liderança e capitalizar a base de clientes já estabelecida, buscando a fidelização.

* **Análise de mercado nos Estados fracos**
  * Realizar estudos de mercado detalhados nos estados com menor volume de vendas (como ND, AZ, WY) para entender as barreiras de entrada, a concorrência e o perfil do consumidor local.

* **Campanhas segmentadas**
  * Desenvolver campanhas de marketing geolocalizadas, adaptando produtos, promoções e mensagens para as características e necessidades de cada estado, especialmente aqueles de baixo desempenho.

* **Otimização logística**
  * Avaliar a eficiência da entrega e os custos logísticos para os estados com poucas vendas. Um serviço de entrega lento ou caro pode ser um fator limitante que afasta novos clientes.

* **Ajuste de oferta de produtos**
  * Analisar se a oferta de produtos é relevante para a demanda e o perfil dos consumidores em cada estado, e considerar a inclusão de produtos específicos para regiões com baixo engajamento.

# 8. Quantos clientes realizaram mais de uma compra?

In [57]:
# Clientes que realizaram mais de uma compra (Corrigido):

consulta("""
SELECT
    cliente_id,
    COUNT(*) AS compras
FROM pedidos_limpos
WHERE status_pedido = 'Concluído'
GROUP BY cliente_id
HAVING compras > 1
ORDER BY compras DESC;
""")

,cliente_id,compras
0,604,5
1,512,5
2,984,4
3,716,4
4,664,4
...,...,...
125,516,2
126,511,2
127,510,2
128,508,2


In [58]:
# Número total de clientes:

consulta("""
SELECT COUNT(*) AS total_clientes
FROM clientes_limpo;
""")

,total_clientes
0,500


*   **Principais Achados**
    *   **Clientes recorrentes**: Dos 500 clientes, **130** realizaram mais de uma compra concluída, o que representa exatamente **26%** do total de clientes cadastrados. Este é um indicador importante de fidelidade de base.

    *   **Potencial de engajamento**: A existência de clientes fiéis que realizaram até 6 compras concluídas demonstra um núcleo de usuários altamente satisfeitos que geram receita recorrente para o marketplace.

    *   **Oportunidade com clientes de uma compra**: Há 370 clientes (74%) que realizaram apenas uma compra concluída ou não possuem compras concluídas no período, indicando uma grande oportunidade para ações de recompra e retenção.

    *   **Foco na conclusão**: Como a taxa de cancelamento geral é alta, muitos clientes que tentaram comprar mais de uma vez acabaram tendo seus pedidos cancelados, reforçando que o gargalo pode estar na conversão final e logística, e não na intenção de compra.

*   **Recomendações Estratégicas**
*   **Fortalecer programas de fidelidade**
    *   Expandir ou aprimorar programas de fidelidade existentes para recompensar ainda mais os clientes que realizam múltiplas compras concluídas, incentivando a recorrência e elevando o número de clientes leais.

*   **Estratégias de reengajamento e conversão para clientes de uma compra**
    *   Desenvolver campanhas de e-mail marketing personalizadas, ofertas especiais ou pesquisas de satisfação direcionadas aos clientes que fizeram apenas uma compra ou cujos pedidos foram cancelados. Entender os motivos do não retorno ou do cancelamento é crucial.

*   **Identificar e nutrir clientes de alto valor**
    *   Criar segmentos específicos para os clientes com maior número de compras concluídas. Oferecer atendimento premium, acesso antecipado a lançamentos ou ofertas exclusivas para fortalecer o relacionamento e a lealdade.

*   **Personalização da experiência**
    *   Utilizar os dados de histórico de compras concluídas para oferecer recomendações de produtos e promoções altamente personalizadas, aumentando a relevância para cada cliente e impulsionando novas vendas.

*   **Otimizar o processo de conclusão de pedidos**
    *   Revisar e otimizar a jornada do cliente desde o carrinho até a finalização da compra para reduzir o número de cancelamentos e aumentar a taxa de conversão para todos os métodos de pagamento.

In [59]:
# Fechando a conexão

conn.close()